# 01 - Train and evaluate

builds the sign-language (ArSL Mosl_alphabet) datasets (80/20 train/val + separate test folder), trains the proposed
PPG-SwinT model and the three baselines, and shows a metrics table.

**Set `DATA_ROOT` below** to the SLR folder that contains `Mosl_alphabet_train/` and `Mosl_alphabet_test/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('ppg_slr'))
import torch
from config import Config
from data.slr import make_datasets
from models.backbone import SwinFeatureExtractor
from models.baselines import build_model
from engine.trainer import train
from train import set_seed

## 1. Configure

In [ ]:
DATA_ROOT = '/kaggle/input/datasets/youssefnouiouar1/sing-language-recognition/SLR'   # <-- EDIT THIS (parent of Mosl_alphabet_train/ and _test/)

cfg = Config()
cfg.data_root = DATA_ROOT
cfg.epochs = 30            # lower to e.g. 5 for a fast first look
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(cfg.seed)
print('device:', cfg.device)

## 2. Build datasets (patient-level split)
The split is by patient, not by image - the loader asserts no patient
leaks between train/val/test.

In [ ]:
datasets = make_datasets(cfg)
print({k: len(v) for k, v in datasets.items()})

## 3. Train the proposed model
`pretrained=True` downloads Swin-Tiny weights on first run (needs
internet). Set `cfg.pretrained = False` to skip.

In [ ]:
cfg.exp_name = 'ppg_swint'
backbone = SwinFeatureExtractor(cfg.backbone, pretrained=cfg.pretrained,
                                stage=cfg.feature_stage)
model = build_model(cfg, backbone)
model, ppg_metrics = train(model, datasets, cfg)
ppg_metrics

## 4. Train the baselines
Each isolates one claim: `plain` (is any machinery needed?), `proto`
(does the probabilistic part beat plain prototypes?), `mcdropout`
(does the prototype part add anything over uncertainty?).

In [ ]:
import copy
results = {'ppg_swint': ppg_metrics}
trained = {'ppg_swint': model}          # keep the models for explainability
for exp in ['plain', 'proto', 'mcdropout']:
    c = copy.deepcopy(cfg); c.exp_name = exp
    set_seed(c.seed)
    bb = SwinFeatureExtractor(c.backbone, pretrained=c.pretrained, stage=c.feature_stage)
    mm = build_model(c, bb)
    mm, results[exp] = train(mm, datasets, c)
    trained[exp] = mm
results

## 5. Metrics table

In [ ]:
import pandas as pd
df = pd.DataFrame(results).T[['accuracy', 'auc', 'f1', 'ece', 'nll']]
df = df.rename_axis('model').round(4)
df

## 6. Explainability evaluation

Accuracy is only half the claim - PPG is meant to be *interpretable*. This
section evaluates the explanations of every model, both quantitatively and
visually.

**Two model families:**
- *Interpretable-by-design* (`ppg_swint`, `proto`): the explanation is the
  prototype attention map - it comes for free from the mechanism.
- *Black boxes* (`plain`, `mcdropout`): no built-in explanation, so we fall
  back to a post-hoc gradient saliency. **Needing an external method to be
  explained at all is itself the finding.**

**Mask-free faithfulness metrics** (slr has no lesion boxes):
- `attention_entropy` - how focused the explanation is (**lower = sharper**).
- `deletion_auc` - remove the most-important pixels; faithful maps drop the
  true-class probability fast (**lower = more faithful**).
- `insertion_auc` - add back the top pixels to a blurred image; faithful
  maps raise probability fast (**higher = more faithful**).

In [ ]:
from engine.explainability import explainability_report, overlay_grid
from engine.trainer import make_loaders

# `trained` (all four models) was populated in section 4 - reuse it directly.
print('models available for explainability:', list(trained))

### 6a. Quantitative explainability table

In [ ]:
loaders = make_loaders(datasets, cfg)
expl = {name: explainability_report(m, loaders['test'], cfg, max_batches=8)
        for name, m in trained.items()}
expl_df = pd.DataFrame(expl).T[['explanation_type', 'attention_entropy',
                                'deletion_auc', 'insertion_auc']]
expl_df = expl_df.rename_axis('model')
for col in ['attention_entropy', 'deletion_auc', 'insertion_auc']:
    expl_df[col] = expl_df[col].astype(float).round(4)
expl_df

**How to read it.** If PPG is doing its job, it should have the *lowest*
attention entropy (sharpest, most localised prototype), a *low* deletion-AUC
and a *high* insertion-AUC (its highlighted region genuinely drives the
decision), while `plain`/`mcdropout` rely on a post-hoc map that is usually
more diffuse and less faithful. If PPG is *not* better here, that is an
honest negative result worth reporting - interpretability is the whole
selling point, so it must be measured, not assumed.

### 6b. Visual comparison (prototype / saliency overlays)

In [ ]:
import matplotlib.pyplot as plt
# grab one batch of test images and overlay each model's explanation
xb, yb = next(iter(loaders['test']))
fig = overlay_grid(trained, xb, yb, cfg, n=4)
fig.savefig('explainability_overlays.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved explainability_overlays.png')

Each row is a test image; each column overlays one model's explanation.
For `ppg_swint`/`proto` this is the upsampled attention of the most-activated
prototype for the predicted class - the region the model says 'looks like a
learned malignant/benign pattern'. For `plain`/`mcdropout` it is a post-hoc
gradient map. Look for whether PPG lands on coherent tissue structures rather
than scattered pixels.

## 7. Grad-CAM heatmaps on the test set (one image per class)

A post-hoc **Grad-CAM** view of the trained model: for each of the 32 classes we
take one test image and overlay the class-discriminative heatmap. The CAM targets
the Swin backbone's final feature map (`model.backbone`, already `[B,C,H,W]`), so
no `reshape_transform` is needed. Switch `GRADCAM_MODEL` to any key in `trained`.

In [ ]:
# Grad-CAM (post-hoc) on one test image per class -> 32 heatmaps.
try:
    from pytorch_grad_cam import GradCAM
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "grad-cam"])
    from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


class _LogitsOnly(nn.Module):
    """Grad-CAM expects model(x) -> logits, but our models return a dict."""
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return self.m(x)["logits"]


GRADCAM_MODEL = "ppg_swint"                 # any key in `trained`
model = trained[GRADCAM_MODEL].eval().to(cfg.device)
wrapped = _LogitsOnly(model)
target_layers = [model.backbone]            # SwinFeatureExtractor -> NCHW feature map

# one test image per class (index order = class label)
test_ds = datasets["test"]
first_of_class = {}
for idx in range(len(test_ds)):
    first_of_class.setdefault(int(test_ds.labels[idx]), idx)
idxs = [first_of_class[c] for c in sorted(first_of_class)]

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
cam = GradCAM(model=wrapped, target_layers=target_layers)

cols = 8
rows = math.ceil(len(idxs) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.0, rows * 2.2))
axes = np.atleast_2d(axes)
for i, idx in enumerate(idxs):
    x, y = test_ds[idx]
    xin = x.unsqueeze(0).to(cfg.device)
    with torch.no_grad():
        pred = int(wrapped(xin).argmax(1))
    heat = cam(input_tensor=xin, targets=[ClassifierOutputTarget(pred)])[0]   # [H,W] in [0,1]
    img = (x.permute(1, 2, 0).cpu().numpy() * std + mean).clip(0, 1)
    ax = axes[i // cols, i % cols]
    ax.imshow(img)
    ax.imshow(heat, cmap="jet", alpha=0.5)
    ax.axis("off")
    mark = "OK" if pred == y else "X"
    ax.set_title(f"{cfg.class_names[y]}->{cfg.class_names[pred]} {mark}", fontsize=6)
for j in range(len(idxs), rows * cols):
    axes[j // cols, j % cols].axis("off")
fig.suptitle(f"Grad-CAM ({GRADCAM_MODEL}) - one test image per class", y=1.002)
plt.tight_layout()
fig.savefig("gradcam_test_per_class.png", dpi=130, bbox_inches="tight")
plt.show()
print("saved gradcam_test_per_class.png")